# 🏛️ Project 4: Sovereign AI Engine — Colab Fine-Tuning
### Fine-Tune Open-Weight LLMs (Unsloth + PEFT QLoRA) on BillSum & Export for vLLM / Local Serving

This notebook runs on a **Free Google Colab T4 GPU (15GB VRAM)**.

**Workflow:**
1. Setup Unsloth & CUDA dependencies
2. Download & preprocess the **BillSum** congressional dataset
3. Baseline Zero-Shot Prompting vs. Fine-Tuned Model
4. Train LoRA / QLoRA adapters with Unsloth
5. Export merged weights (or 4-bit GGUF for local CPU inference)

In [ ]:
# Step 1: Install Unsloth, Hugging Face PEFT, TRL, and dependencies
!pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes datasets rouge-score
!pip install xformers "torch>=2.4.0"

In [ ]:
# Step 2: Verify GPU
!nvidia-smi

In [ ]:
# Step 3: Load Model with Unsloth FastLanguageModel (4-bit QLoRA)
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
# Choose Qwen 2.5 7B or Llama 3.2 3B
model_name = "Qwen/Qwen2.5-7B-Instruct"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    dtype=None,
    load_in_4bit=True,
)

print("Model loaded successfully in 4-bit!")

In [ ]:
# Step 4: Attach PEFT LoRA Adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

model.print_trainable_parameters()

In [ ]:
# Step 5: Download & Format BillSum Dataset
from datasets import load_dataset

dataset = load_dataset("billsum", split="train[:1500]")

prompt_template = """<|im_start|>system
You are an expert legislative analyst and summarizer. Provide a concise, accurate, and structured summary of the congressional bill below.<|im_end|>
<|im_start|>user
Summarize this bill:

{text}<|im_end|>
<|im_start|>assistant
{summary}<|im_end|>"""

def formatting_prompts_func(examples):
    texts = []
    for text, summary in zip(examples["text"], examples["summary"]):
        texts.append(prompt_template.format(text=text[:3500], summary=summary))
    return {"text": texts}

formatted_dataset = dataset.map(formatting_prompts_func, batched=True)
print(f"Prepared {len(formatted_dataset)} samples for training.")

In [ ]:
# Step 6: Train with SFTTrainer
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=formatted_dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        max_steps=120,
        learning_rate=2e-4,
        fp16=not FastLanguageModel.is_bfloat16_supported(),
        bf16=FastLanguageModel.is_bfloat16_supported(),
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=42,
        output_dir="outputs",
    ),
)

trainer_stats = trainer.train()

In [ ]:
# Step 7: Test Inference with the Fine-Tuned Model
FastLanguageModel.for_inference(model)

test_prompt = """<|im_start|>system
You are an expert legislative analyst and summarizer. Provide a concise, accurate, and structured summary of the congressional bill below.<|im_end|>
<|im_start|>user
Summarize this bill:

SECTION 1. SHORT TITLE. This Act may be cited as the 'Clean Energy Innovation Act of 2026'. SEC. 2. ESTABLISHMENT OF GRANT PROGRAM. The Secretary of Energy shall establish a competitive grant program to support advanced solar and storage research at eligible academic institutions.<|im_end|>
<|im_start|>assistant
"""

inputs = tokenizer([test_prompt], return_tensors="pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens=150, temperature=0.2, use_cache=True)
print(tokenizer.batch_decode(outputs)[0])

In [ ]:
# Step 8: Save LoRA Adapters and GGUF for Local Serving
# 1. Save LoRA adapter locally
model.save_pretrained("sovereign_lora_adapter")
tokenizer.save_pretrained("sovereign_lora_adapter")

# 2. Export GGUF for local CPU execution via llama.cpp or Ollama
# model.save_pretrained_gguf("sovereign_q4_k_m", tokenizer, quantization_method="q4_k_m")

print("Model saved! Download 'sovereign_lora_adapter' folder to use in your local project.")